Data Collator是HuggingFace开源的transformers模块进行数据处理的重要部分。它的输入是由数据集元素组成的列表，将其组装成批次，其中数据集元素为相同数据类型的train_dataset或者eval_dataset。

为了组装成数据批次，Data collators会应用某些处理（比如padding），有些（比如DataCollatorForLanguageModeling）还会在数据批次上应用随机数据增强（比如随机masking）。

Data collators是为了特定任务而设计的，如下：

- Causal language modeling (CLM)

- Masking language modeling (MLM)

- Sequence classification

- Seq2Seq

- Token classification

以sequence classification任务为例，data collator只需将所有序列填充到一个小批次中，以确保它们具有相同的长度。

## DataCollatorWithPadding
DataCollatorWithPadding类将输入的input_ids、attention_mask等向量做padding处理，截断或补充padding，使得它们的长度保持统一。

In [ ]:
from transformers import DataCollatorWithPadding, AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen1.5-7B")
# DataCollatorWithPadding(tokenizer, max_length=10, padding='max_length')
dc = DataCollatorWithPadding(tokenizer)
raw_tokens = tokenizer(['南京', '南京市长', '南京市长江大桥'])
print(raw_tokens)
print(dc(raw_tokens))

"""
{'input_ids': [[102034], [102034, 102975], [102034, 102975, 69177, 106936]], 'attention_mask': [[1], [1, 1], [1, 1, 1, 1]]}
{'input_ids': tensor([[102034, 151643, 151643, 151643],
        [102034, 102975, 151643, 151643],
        [102034, 102975,  69177, 106936]]), 'attention_mask': tensor([[1, 0, 0, 0],
        [1, 1, 0, 0],
        [1, 1, 1, 1]])}
"""

# DataCollatorForTokenClassification
DataCollatorForTokenClassification类适用于序列标注任务（token classification），比如命名实体识别（NER），每个token都会对应一个预测label，当一个序列的token比label多时，对额外的label位置添加-100处理，使得在计算交叉熵损失时，-100位置的label损失为0不用考虑。

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen1.5-7B")
dc = DataCollatorForTokenClassification(tokenizer)
data = [('南京', [1]), ('南京市长', [1, 0]), ('南京市长江大桥', [1, 0])]
features = []
for text, label in data:
    feature = tokenizer(text)
    feature['label'] = label
    features.append(feature)

print(dc(features))

"""
{'input_ids': tensor([[102034, 151643, 151643, 151643],
        [102034, 102975, 151643, 151643],
        [102034, 102975,  69177, 106936]]), 'attention_mask': tensor([[1, 0, 0, 0],
        [1, 1, 0, 0],
        [1, 1, 1, 1]]), 'label': tensor([[   1, -100, -100, -100],
        [   1,    0, -100, -100],
        [   1,    0, -100, -100]])}
"""

## DataCollatorForSeq2Seq
DataCollatorForSeq2Seq类适用于Seq2Seq任务，包括机器翻译，文本生成任务等。该类根据前面的序列来预测后面序列，后面序列为labels，一个batch内多条后面序列长度不同时，用-100来填充。同样地，在计算损失时，-100位置的损失不予考虑。

In [ ]:
from transformers import DataCollatorForSeq2Seq, AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen1.5-7B")
dc = DataCollatorForSeq2Seq(tokenizer)
data = [('南京', '市长江大桥'), ('南京市', '长江大桥'), ('南京市长江', '大桥')]

features = []
for text, label in data:
    feature = tokenizer(text)
    feature['labels'] = tokenizer(label)['input_ids']
    features.append(feature)

print(dc(features))

"""
{'input_ids': tensor([[102034, 151643, 151643],
        [112891, 151643, 151643],
        [102034, 102975,  69177]]), 'attention_mask': tensor([[1, 0, 0],
        [1, 0, 0],
        [1, 1, 1]]), 'labels': tensor([[102975,  69177, 106936],
        [104924, 106936,   -100],
        [106936,   -100,   -100]])}
"""

## DataCollatorForLanguageModeling
DataCollatorForLanguageModeling类适用于语言模型，比如BERT系列等。

该类有个mlm参数，其默认值为True, 取值如下：

- 如果设置为False，则labels与inputs一致，使用-100进行填充

- 如果设置为True，则labels对于non-masked tokens设置为-100，对于masked token，其处理方式同BERT模型：80% MASK（MASK值）, 10% random（随机替换成其它token值）, 10% original（原始值）。

mlm_probability参数默认值为0.15，即对input_ids中的token会有15%的概率进行mask。

In [ ]:
from transformers import DataCollatorForLanguageModeling, AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-chinese")
dc = DataCollatorForLanguageModeling(tokenizer)
data = ['南京', '南京市', '南京市长江']
raw_tokens = [tokenizer(text) for text in data]
print(dc(raw_tokens))

"""
{'input_ids': tensor([[ 101, 1298,  776,  102,    0,    0,    0],
        [ 101, 1298,  776, 2356,  102,    0,    0],
        [ 101, 1298,  776, 2356, 7270, 3736,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 0, 0, 0],
        [1, 1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 1, 1]]), 'labels': tensor([[-100, -100, -100, -100, -100, -100, -100],
        [-100, -100, -100, -100, -100, -100, -100],
        [-100, -100,  776, -100, -100, -100, -100]])}
"""